# Adaptive Market Intelligence — V1 Baseline

**PROVE BEFORE TRADE**

This experiment uses real daily OHLCV data and a strictly temporal split. No random shuffling. The target is next-day UP/DOWN direction.

In [ ]:
!pip -q install -r ../requirements.txt

import sys
sys.path.append('../')

from src.data_loader import load_ohlcv
from src.features import build_features, FEATURE_COLUMNS
from src.models import build_models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

SYMBOL = 'BTC-USD'
df = load_ohlcv(SYMBOL, start='2018-01-01')
data = build_features(df)
print(data.shape)
data.tail()

In [ ]:
# 70/15/15 chronological split
n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end]
val = data.iloc[train_end:val_end]
test = data.iloc[val_end:]

X_train, y_train = train[FEATURE_COLUMNS], train['target_next_day_up']
X_val, y_val = val[FEATURE_COLUMNS], val['target_next_day_up']
X_test, y_test = test[FEATURE_COLUMNS], test['target_next_day_up']

print('Train:', train.index.min(), '→', train.index.max())
print('Validation:', val.index.min(), '→', val.index.max())
print('Test:', test.index.min(), '→', test.index.max())

In [ ]:
results = []
models = build_models()
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
results_df

## Interpretation

Accuracy above 50% alone is **not** proof of a tradable edge. The next stages must include walk-forward validation, realistic transaction costs/slippage, a proper backtest, and paper trading.